
Computes the LST/NDVI trajectory (NHDA vs. RA) including the difference for A
SINGLE NHDA directly from the raster data (analogous to the LST_/NDVI_Comparison
scripts), instead of reading from already aggregated columns.

Geometries: from COMBINED_GPKG, row with type == 'NHDA' or type == 'RA'
for the selected nhda_id.

For each year:
  - n_random_points random points within the NHDA and RA geometry
  - Extract LST/NDVI at these points from the respective year's raster
  - Median and std per group (NHDA, RA)
  - Difference = Median(NHDA) - Median(RA)
  - Std of the difference via bootstrap (resampling of the points, see below)

In [ ]:

import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import Point
import random
import re
import os

# ============================================================================
# CONFIGURATION
# ============================================================================
TARGET_NHDA_ID = "09186_5"

COMBINED_GPKG = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI.gpkg"
CONSTRUCTION_GPKG = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\New_Housing_Development_Areas\NHDA_with_construction_years_RF_AUC.gpkg"

# --- Raster sources: TODO adjust to current EO4CAM paths if needed ---
LST_DIR = Path(r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\processed_datasets\LST_Landsat")
LST_YEARS = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
LST_FILE_TEMPLATE = "Landsat89_L2_LST_JJA_Median_Bayern_{year}.tif"

NDVI_DIR = Path(r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\processed_datasets\Sentinel_2_WASP_NDVI")  # TODO check path
NDVI_PATTERN = "Bayern_NDVI_*median_25832.tif"

OUTPUT_DIR = r"C:\Users\agz90fk\Documents\EO4CAM\07_Abbildungen\Masterarbeit\Single_NHDA_Plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_RANDOM_POINTS = 100
N_BOOTSTRAP = 1000
MIN_VALID_FRACTION = 0.3  # min. fraction of valid (non-nodata) values, otherwise the year is skipped
RANDOM_SEED = 42

CONSTRUCTION_SPECIAL_MAP = {
    'AUC_2015': 2014,
    'AUC_2016': 2015,
}

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def generate_random_points_in_geometry(geometry, n_points, max_attempts=None):
    if max_attempts is None:
        max_attempts = n_points * 100
    points = []
    attempts = 0
    minx, miny, maxx, maxy = geometry.bounds
    while len(points) < n_points and attempts < max_attempts:
        p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if geometry.contains(p):
            points.append(p)
        attempts += 1
    return points


def extract_raster_values(points_gdf, raster_path, is_ndvi=False):
    """Extracts raster values at points. NDVI is scaled from int(-100..100) to -1..1."""
    try:
        with rasterio.open(raster_path) as src:
            points_reproj = points_gdf.to_crs(src.crs) if points_gdf.crs != src.crs else points_gdf
            values = []
            for _, point in points_reproj.iterrows():
                coords = [(point.geometry.x, point.geometry.y)]
                for val in src.sample(coords):
                    raw = float(val[0])
                    if src.nodata is not None and raw == src.nodata:
                        values.append(np.nan)
                    elif is_ndvi and (raw < -100.0 or raw > 100.0):
                        values.append(np.nan)
                    elif is_ndvi:
                        values.append(raw / 100.0)
                    else:
                        values.append(raw)
            return np.array(values)
    except Exception as exc:
        print(f"      ! Error reading {raster_path}: {exc}")
        return np.full(len(points_gdf), np.nan)


def bootstrap_diff_std(nhda_values, ra_values, n_bootstrap=N_BOOTSTRAP, rng=None):
    """Std of the difference of the medians via bootstrap resampling of the raw points."""
    if rng is None:
        rng = np.random.default_rng(RANDOM_SEED)
    nhda_values = nhda_values[~np.isnan(nhda_values)]
    ra_values = ra_values[~np.isnan(ra_values)]
    if len(nhda_values) < 2 or len(ra_values) < 2:
        return np.nan
    diffs = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        boot_nhda = rng.choice(nhda_values, size=len(nhda_values), replace=True)
        boot_ra = rng.choice(ra_values, size=len(ra_values), replace=True)
        diffs[i] = np.median(boot_nhda) - np.median(boot_ra)
    return float(np.std(diffs))


def to_long(records):
    df = pd.DataFrame(records)
    return df.dropna(subset=['value'])


# ============================================================================
# 1. LOAD GEOMETRIES AND CONSTRUCTION START
# ============================================================================
print("=" * 80)
print(f"SINGLE-NHDA TRAJECTORY (from raw data): {TARGET_NHDA_ID}")
print("=" * 80)

for path in [COMBINED_GPKG, CONSTRUCTION_GPKG]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

gdf = gpd.read_file(COMBINED_GPKG)
gdf['nhda_id'] = gdf['nhda_id'].astype(str)

subset = gdf[gdf['nhda_id'] == TARGET_NHDA_ID].copy()
if subset.empty:
    raise ValueError(f"NHDA ID {TARGET_NHDA_ID} not found in Combined GPKG.")

type_col = None
for candidate in ['type', 'area_type']:
    if candidate in subset.columns:
        type_col = candidate
        break
if type_col is None:
    raise ValueError("No type column ('type' or 'area_type') found.")

nhda_rows = subset[subset[type_col].astype(str) == 'NHDA']
ra_rows = subset[subset[type_col].astype(str) == 'RA']
if nhda_rows.empty or ra_rows.empty:
    raise ValueError(f"NHDA or RA geometry missing for {TARGET_NHDA_ID} in Combined GPKG.")

nhda_geom = nhda_rows.iloc[0].geometry
ra_geom = ra_rows.iloc[0].geometry
crs = gdf.crs

print(f"   NHDA area: {nhda_geom.area / 10000:.2f} ha")
print(f"   RA area:   {ra_geom.area / 10000:.2f} ha")

gdf_const = gpd.read_file(CONSTRUCTION_GPKG)
id_candidates = ['nhda_id', 'cluster_id', 'nda_id']
id_col = next((c for c in id_candidates if c in gdf_const.columns), None)
if id_col is None:
    raise ValueError(f"No ID column found in Construction GPKG. Tried: {id_candidates}")
if 'construction_start_year' not in gdf_const.columns:
    raise ValueError("Column 'construction_start_year' not found in Construction GPKG.")

gdf_const['nhda_id'] = gdf_const[id_col].astype(str)
const_row = gdf_const[gdf_const['nhda_id'] == TARGET_NHDA_ID]
if const_row.empty:
    raise ValueError(f"NHDA ID {TARGET_NHDA_ID} not found in Construction GPKG.")

raw_construction = str(const_row.iloc[0]['construction_start_year']).strip()
construction_year = CONSTRUCTION_SPECIAL_MAP.get(raw_construction, raw_construction)
construction_year = float(construction_year)

print(f"   Construction start (raw):  {raw_construction}")
print(f"   Construction start (used): {construction_year}")

# ============================================================================
# 2. GENERATE RANDOM POINTS (once per geometry, same for all years)
# ============================================================================
random.seed(RANDOM_SEED)
nhda_points = generate_random_points_in_geometry(nhda_geom, N_RANDOM_POINTS)
ra_points = generate_random_points_in_geometry(ra_geom, N_RANDOM_POINTS)

if len(nhda_points) < N_RANDOM_POINTS * 0.5 or len(ra_points) < N_RANDOM_POINTS * 0.5:
    raise ValueError("Too few valid random points generated (geometry may be too small/complex).")

nhda_points_gdf = gpd.GeoDataFrame(geometry=nhda_points, crs=crs)
ra_points_gdf = gpd.GeoDataFrame(geometry=ra_points, crs=crs)

print(f"   Random points: NHDA={len(nhda_points)}, RA={len(ra_points)}")

rng = np.random.default_rng(RANDOM_SEED)

# ============================================================================
# 3. LST: EXTRACT YEAR BY YEAR FROM RASTERS
# ============================================================================
print("\n--- LST ---")
lst_records_nhda, lst_records_ra, lst_records_diff = [], [], []

for year in LST_YEARS:
    lst_file = LST_DIR / LST_FILE_TEMPLATE.format(year=year)
    if not lst_file.exists():
        print(f"   {year}: file not found, skipped ({lst_file.name})")
        continue

    nhda_vals = extract_raster_values(nhda_points_gdf, lst_file, is_ndvi=False)
    ra_vals = extract_raster_values(ra_points_gdf, lst_file, is_ndvi=False)

    nhda_valid = nhda_vals[~np.isnan(nhda_vals)]
    ra_valid = ra_vals[~np.isnan(ra_vals)]

    if len(nhda_valid) < N_RANDOM_POINTS * MIN_VALID_FRACTION or len(ra_valid) < N_RANDOM_POINTS * MIN_VALID_FRACTION:
        print(f"   {year}: too few valid values (n_nhda={len(nhda_valid)}, n_ra={len(ra_valid)}), skipped")
        continue

    nhda_median, nhda_std = np.median(nhda_valid), np.std(nhda_valid)
    ra_median, ra_std = np.median(ra_valid), np.std(ra_valid)
    diff = nhda_median - ra_median
    diff_std = bootstrap_diff_std(nhda_valid, ra_valid, rng=rng)

    print(f"   {year}: NHDA={nhda_median:.2f}±{nhda_std:.2f}  RA={ra_median:.2f}±{ra_std:.2f}  "
          f"Diff={diff:+.2f}±{diff_std:.2f}  (n_nhda={len(nhda_valid)}, n_ra={len(ra_valid)})")

    lst_records_nhda.append({'year': year, 'value': nhda_median, 'std': nhda_std})
    lst_records_ra.append({'year': year, 'value': ra_median, 'std': ra_std})
    lst_records_diff.append({'year': year, 'value': diff, 'std': diff_std})

lst_nhda = to_long(lst_records_nhda)
lst_ra = to_long(lst_records_ra)
lst_diff = to_long(lst_records_diff)

# ============================================================================
# 4. NDVI: FIND AVAILABLE YEARS AND EXTRACT
# ============================================================================
print("\n--- NDVI ---")
ndvi_files = sorted(NDVI_DIR.glob(NDVI_PATTERN)) if NDVI_DIR.exists() else []
ndvi_map = {}
for f in ndvi_files:
    m = re.search(r'Bayern_NDVI_(\d{4})', f.stem)
    if m:
        ndvi_map[int(m.group(1))] = f

if not ndvi_map:
    print(f"   No NDVI files found in {NDVI_DIR} (pattern: {NDVI_PATTERN}) - check path/pattern!")

ndvi_records_nhda, ndvi_records_ra, ndvi_records_diff = [], [], []

for year in sorted(ndvi_map.keys()):
    ndvi_file = ndvi_map[year]

    nhda_vals = extract_raster_values(nhda_points_gdf, ndvi_file, is_ndvi=True)
    ra_vals = extract_raster_values(ra_points_gdf, ndvi_file, is_ndvi=True)

    nhda_valid = nhda_vals[~np.isnan(nhda_vals)]
    ra_valid = ra_vals[~np.isnan(ra_vals)]

    if len(nhda_valid) < N_RANDOM_POINTS * MIN_VALID_FRACTION or len(ra_valid) < N_RANDOM_POINTS * MIN_VALID_FRACTION:
        print(f"   {year}: too few valid values (n_nhda={len(nhda_valid)}, n_ra={len(ra_valid)}), skipped")
        continue

    nhda_median, nhda_std = np.median(nhda_valid), np.std(nhda_valid)
    ra_median, ra_std = np.median(ra_valid), np.std(ra_valid)
    diff = nhda_median - ra_median
    diff_std = bootstrap_diff_std(nhda_valid, ra_valid, rng=rng)

    print(f"   {year}: NHDA={nhda_median:.3f}±{nhda_std:.3f}  RA={ra_median:.3f}±{ra_std:.3f}  "
          f"Diff={diff:+.3f}±{diff_std:.3f}  (n_nhda={len(nhda_valid)}, n_ra={len(ra_valid)})")

    ndvi_records_nhda.append({'year': year, 'value': nhda_median, 'std': nhda_std})
    ndvi_records_ra.append({'year': year, 'value': ra_median, 'std': ra_std})
    ndvi_records_diff.append({'year': year, 'value': diff, 'std': diff_std})

ndvi_nhda = to_long(ndvi_records_nhda)
ndvi_ra = to_long(ndvi_records_ra)
ndvi_diff = to_long(ndvi_records_diff)

# ============================================================================
# 5. SAVE TABLES
# ============================================================================
export_df = pd.merge(
    lst_nhda.rename(columns={'value': 'LST_nhda', 'std': 'LST_nhda_std'}),
    lst_ra.rename(columns={'value': 'LST_ra', 'std': 'LST_ra_std'}),
    on='year', how='outer'
)
export_df = pd.merge(export_df, lst_diff.rename(columns={'value': 'LST_diff', 'std': 'LST_diff_std'}), on='year', how='outer')
export_df = pd.merge(export_df, ndvi_nhda.rename(columns={'value': 'NDVI_nhda', 'std': 'NDVI_nhda_std'}), on='year', how='outer')
export_df = pd.merge(export_df, ndvi_ra.rename(columns={'value': 'NDVI_ra', 'std': 'NDVI_ra_std'}), on='year', how='outer')
export_df = pd.merge(export_df, ndvi_diff.rename(columns={'value': 'NDVI_diff', 'std': 'NDVI_diff_std'}), on='year', how='outer')
export_df = export_df.sort_values('year')
export_csv = f"{OUTPUT_DIR}/{TARGET_NHDA_ID}_raw_trajectory.csv"
export_df.to_csv(export_csv, index=False)
print(f"\n   Table saved: {export_csv}")

# ============================================================================
# 6. PLOT
# ============================================================================
place_col = next((c for c in ['gemeinde', 'kommune', 'ort', 'place_name', 'location'] if c in subset.columns), None)
place_name = nhda_rows.iloc[0][place_col] if place_col else None
title_place = f" in {place_name}" if place_name else ""

fig, axes = plt.subplots(4, 1, figsize=(9, 16))
fig.suptitle(f"New Housing Development Area ({TARGET_NHDA_ID}){title_place}\n in Pfaffenhofen a.d. Ilm", fontweight='bold', fontsize=16)


def plot_two_lines(ax, series_a, series_b, label_a, label_b, color_a, color_b, ylabel, title):
    for series, label, color in [(series_a, label_a, color_a), (series_b, label_b, color_b)]:
        if series.empty:
            continue
        ax.plot(series['year'], series['value'], marker='o', color=color, label=label, linewidth=2)
        if series['std'].notna().any():
            ax.fill_between(
                series['year'],
                series['value'] - series['std'],
                series['value'] + series['std'],
                color=color, alpha=0.2
            )
    ax.axvline(construction_year, linestyle='--', color='olive', linewidth=1.8, label='Construction start')
    # Light grey axes
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color("#d9d9d9")
        ax.spines[spine].set_linewidth(0.8)

    # Remove top/right border
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Year')
    ax.set_xlim(2015, 2025)
    ax.set_xticks(range(2015, 2026))
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3)
    leg = ax.legend(frameon=True,loc="upper right")
    leg.get_frame().set_edgecolor("#d9d9d9")
    leg.get_frame().set_linewidth(0.8)
    leg.get_frame().set_facecolor("white")

def plot_diff(ax, series, color, ylabel, title, diff_label):
    if not series.empty:
        ax.plot(series['year'], series['value'], marker='o', color=color, linewidth=2, label=diff_label)
        if series['std'].notna().any():
            ax.fill_between(
                series['year'],
                series['value'] - series['std'],
                series['value'] + series['std'],
                color=color, alpha=0.2
            )
    ax.axhline(0, linestyle='--', color='red', linewidth=1.5)
    ax.axvline(construction_year, linestyle='--', color='olive', linewidth=1.8, label='Construction start')
    ax.set_ylabel(ylabel)
    # Light grey axes
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color("#fff5f5")
        ax.spines[spine].set_linewidth(0.8)

    # Remove top/right border
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel('Year')
    ax.set_xlim(2015, 2025)
    ax.set_xticks(range(2015, 2026))
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3)
    leg = ax.legend(frameon=True,loc="upper right")
    leg.get_frame().set_edgecolor("#fffefe")
    leg.get_frame().set_linewidth(0.8)
    leg.get_frame().set_facecolor("white")


plot_two_lines(axes[0], lst_nhda, lst_ra, 'NHDA', 'RA', 'red', 'blue', 'LST [°C]', 'LST')
plot_diff(axes[1], lst_diff, 'darkred', 'ΔLST [°C]', 'ΔLST', 'Difference (LST)')
plot_two_lines(axes[2], ndvi_nhda, ndvi_ra, 'NHDA', 'RA', 'red', 'blue', 'NDVI', 'NDVI')
plot_diff(axes[3], ndvi_diff, 'darkgreen', 'ΔNDVI', 'ΔNDVI', 'Difference (NDVI)')

plt.tight_layout(rect=[0, 0, 1, 0.97])
out_path = f"{OUTPUT_DIR}/{TARGET_NHDA_ID}_trajectory.png"
plt.savefig(out_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"   Plot saved: {out_path}")
print("\nDONE")